In [1]:
# %%
import torch
import os
import sys
from datasets import load_dataset
os.environ["TOKENIZERS_PARALLELISM"] = 'false'
sys.path.insert(0, "..")
from testbed.data import prepare_dataloader
import config
import exp_settings as setting

dataset = load_dataset(
    os.path.join(config.testbed_dir, "data", "vqav2"),
    split="validation",
    data_dir=".",
    images_dir=config.coco_dir,
    trust_remote_code=True,
)

hparams = {
    "batch_size": 8,
    "num_shots": 0,
    "dtype": torch.float16,
    "generate_args": setting.generate_args,
}

dataloader = prepare_dataloader(
    dataset,
    batch_size=hparams["batch_size"],
    num_shots=hparams["num_shots"],
)


In [3]:
# %%
from transformers import BitsAndBytesConfig
from testbed.models import Idefics
from dev.icv_encoder import GlobalICVEncoder, AttnAwareEncoder, AttnPerturbEncoder

device = torch.device("cuda:1")
lmm = Idefics(
    config.idefics_9b_path,
    torch_dtype=torch.float16,
).to(device)
lmm.eval()
icv_encoder = AttnPerturbEncoder(4096, 32, record_hidden_states=True).to(device)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [4]:
# %%
sd = torch.load("../results/ckpt/icv_cpk-attn-layer-wise.pth")
sd = {
    k.removeprefix("icv_encoder."): v.squeeze()
    for k, v in sd.items()
    if k.startswith("icv_encoder.")
}
icv_encoder.load_state_dict(sd, strict=False)


/tmp/ipykernel_1171066/862088364.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load("../results/ckpt/icv_cpk-attn-layer-wise.pth")


<All keys matched successfully>

In [5]:
# %%
from testbed.models.model_base import HookType


hooks = icv_encoder.register_hook_for(lmm)


In [6]:
# %%
from tqdm import tqdm
import evaluate

from testbed.data import prepare_vqa_input
from testbed.data.vqav2 import postprocess_generation

total_acc = evaluate.load("Kamichanw/vqa_accuracy")
result = []

# for simplicity, just run 10 batches
for _, batch in zip(range(100), tqdm(dataloader, desc=f"Evaluating {lmm.model_name} ...")):
    text, images = prepare_vqa_input(
        batch, instruction="Provide an answer to the question. Use the image to answer."
    )
    predictions = lmm.generate(text, images, **hparams["generate_args"])
    for pred, context in zip(predictions, batch):
        last_qa = context[-1]
        gt_answer = [item["answer"] for item in last_qa["answers"]]
        prediction = postprocess_generation(pred)
        total_acc.add(
            prediction=prediction,
            reference=gt_answer,
            question_types=last_qa["question_type"],
            answer_types=last_qa["answer_type"],
        )
        result.append(
            {
                "question_id": last_qa["question_id"],
                "raw_output": pred,
                "question": last_qa["question"],
                "question_type": last_qa["question_type"],
                "answer_type": last_qa["answer_type"],
                "prediction": prediction,
                "answers": last_qa["answers"],
            }
        )

eval_result = total_acc.compute()
print(eval_result)


Evaluating idefics-9b ...:   8%|▊         | 99/1250 [01:33<18:05,  1.06it/s]

{'overall': 48.262500000000045, 'perAnswerType': {'other': 36.51219512195123, 'yes/no': 69.13907284768216, 'number': 31.363636363636363}, 'perQuestionType': {'what is the man': 11.11111111111111, 'are there': 62.5, 'what kind of': 43.04347826086956, 'is the': 67.23076923076923, 'how many': 34.032258064516135, 'what is': 22.0, 'do you': 78.0, 'none of the above': 42.222222222222214, 'what color is the': 44.35483870967742, 'is this an': 100.0, 'is this a': 83.10344827586206, 'is there a': 60.833333333333336, 'is it': 64.375, 'what time': 7.5, 'what is the': 31.500000000000007, 'are the': 69.09090909090911, 'what': 41.612903225806456, 'what animal is': 15.0, 'what type of': 52.857142857142854, 'how': 15.714285714285717, 'does the': 65.45454545454544, 'where is the': 0.0, 'how many people are': 57.49999999999999, 'what color': 83.33333333333333, 'what is on the': 23.636363636363637, 'are they': 55.0, 'is this': 61.29032258064516, 'are': 60.76923076923077, 'what brand': 100.0, 'is there': 4

In [7]:
# %%
hparams["dtype"] = str(hparams["dtype"])
evaluate.save("./", eval_result=eval_result, hparams=hparams, records=result)


PosixPath('result-2024_09_16-23_15_56.json')